# 01 — Ingest: BLS Unemployment Data

Fetches U.S. unemployment data from the Bureau of Labor Statistics (BLS)
Public Data API v2 and lands it in `data/raw/` and `data/project.duckdb`.

**Two data pulls:**
1. **National demographic series** — monthly unemployment rate by age, sex,
   race/ethnicity, and alternative measures U-3 through U-6 (1948–2026).
   20 BLS CPS series (LNS prefix), seasonally adjusted.
2. **State-level series** — monthly unemployment rate for all 50 states + DC
   (1976–2026). 51 BLS LAUS series (LASST prefix), seasonally adjusted.

**API notes:** BLS Public Data API v2, no registration key required for these
pull sizes. Without a key: 25 series/request, 10-year windows, 25 req/day.
Both pulls fit comfortably within those limits.

**No transformation happens here** — raw observations land in DuckDB exactly
as the BLS API returns them. All cleaning is in `02-clean.ipynb`.


In [ ]:
import sys, os
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched
PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from ingest import (
    load_config,
    ingest_bls_api,
    KEY_UNEMPLOYMENT_SERIES,
    ALTERNATIVE_MEASURES_SERIES,
    STATE_UNEMPLOYMENT_SERIES,
)
from clean_quality import get_connection, load_to_duckdb, register_source
import pandas as pd
import duckdb

cfg = load_config()
print(f"Project: {cfg['project_name']}")
print(f"DuckDB:  {cfg['settings']['duckdb_file']}")


## National Demographic Series (BLS CPS, LNS prefix)

Source: **BLS Current Population Survey (CPS)**, the monthly household survey
that produces the headline unemployment rate.

**Series fetched (21 total):**

| Category | Series |
|---|---|
| Overall | Total 16+, **Total 20+ (adults)**, Men 16+, Women 16+ |
| By sex (adults) | Men 20+, Women 20+ |
| By age | 16–19, 20–24, 25–34, 35–44, 45–54 |
| By race/ethnicity | White, Black, Hispanic, Asian |
| Alternative measures | U-3 (official), U-4, U-5, U-6 (broadest) |

All series are **seasonally adjusted monthly rates** (unemployment %).

**Definition (U-3, the headline rate):** unemployed = without a job, available
to work, and actively looked in the last 4 weeks ÷ civilian labor force. U-6
is the broadest measure — adds marginally attached workers and people working
part-time for economic reasons. Never compare U-3 and U-6 as if they're the
same thing.

**Series breaks to note:**
- U-4/U-5/U-6 alternative measures only go back to 1994 (introduced then).
- Black series back to 1972, Hispanic to 1973, Asian to 2003.
- White series back to 1954; total/sex/most age groups back to 1948.
- Two older age-bracket series (55-64y, 65+y) returned no data on this API —
  those BLS series IDs may have been replaced; not a critical gap for the core
  story and can be filled in later.


In [ ]:
# Load from the cached raw parquet (already fetched this session).
# The parquet was saved to data/raw/bls_api_data.parquet by ingest_bls_api().
# Re-running this cell is safe: it just reloads from cache, no API call.
df_national = pd.read_parquet("data/raw/bls_api_data.parquet")
df_national["label"] = df_national["series_id"].map(
    {**KEY_UNEMPLOYMENT_SERIES, **ALTERNATIVE_MEASURES_SERIES}
)

print(f"National series loaded: {len(df_national):,} observations")
print(f"Series present: {df_national['series_id'].nunique()} of 20 requested")
print()

# Year coverage by series
coverage = (
    df_national
    .groupby(["label", "series_id"])["year"]
    .agg(first="min", last="max", n="count")
    .reset_index()
    .sort_values("first")
)
print(coverage[["label", "first", "last", "n"]].to_string(index=False))


### New series — Total, 20 years & over (adults): `LNS14000024`

Added so Chart 4 in `04-viz` can compare teens (16–19) against a **real
adults-20+ total**, rather than the overall 16+ rate (which includes teens) or
an unweighted men/women workaround. `LNS14000024` is the BLS CPS
seasonally-adjusted unemployment rate for workers **20 years and over**, the
genuine combined-sex adult total — full coverage **1948–2026**.

Fetched via the same BLS Public Data API v2 pattern used for every other series
(`ingest_bls_api`, keyless pull; a `BLS_API_KEY` in the gitignored `.env` raises
the limits but isn't required at this size). The cell below is **idempotent** —
it only hits the API if `LNS14000024` isn't already in the cached raw parquet,
then merges it in and reloads `df_national`.

In [ ]:
# Ensure the 20+ adult total (LNS14000024) is in the raw parquet (idempotent).
SID_20PLUS = "LNS14000024"
_pq = "data/raw/bls_api_data.parquet"
_raw = pd.read_parquet(_pq)
if not (_raw["series_id"] == SID_20PLUS).any():
    print(f"{SID_20PLUS} missing from raw parquet — fetching from BLS API...")
    _orig = _raw.copy()
    _new = ingest_bls_api(cfg, [SID_20PLUS], start_year=1948, end_year=2026)
    _new = _new[_new["series_id"] == SID_20PLUS].copy()
    # ingest_bls_api overwrites the parquet with ONLY the fetched series, so
    # re-merge against the full original set and rewrite the parquet.
    _merged = pd.concat([_orig, _new], ignore_index=True)
    _merged = (_merged.drop_duplicates(["series_id", "year", "period"])
                      .sort_values(["series_id", "year", "period"])
                      .reset_index(drop=True))
    _merged.to_parquet(_pq, index=False)
    print(f"  merged: parquet now {len(_merged):,} rows, {SID_20PLUS} present.")
else:
    print(f"{SID_20PLUS} already in raw parquet — no API call.")

# Reload df_national from the (now complete) parquet with fresh labels.
df_national = pd.read_parquet(_pq)
df_national["label"] = df_national["series_id"].map(
    {**KEY_UNEMPLOYMENT_SERIES, **ALTERNATIVE_MEASURES_SERIES}
)
print(f"df_national: {len(df_national):,} obs, {df_national['series_id'].nunique()} series")
_cov = df_national[df_national["series_id"] == SID_20PLUS]["year"].agg(["min", "max", "count"])
print(f"  {SID_20PLUS} (Total, 20+): {int(_cov['min'])}-{int(_cov['max'])}, {int(_cov['count'])} obs")

In [ ]:
con = get_connection(cfg)

# Load national data
con.execute("DROP TABLE IF EXISTS unemployment_national")
con.execute("CREATE TABLE unemployment_national AS SELECT * FROM df_national ORDER BY series_id, year, period")

n = con.execute("SELECT COUNT(*) FROM unemployment_national").fetchone()[0]
print(f"unemployment_national: {n:,} rows loaded into DuckDB")

register_source(
    con,
    "unemployment_national",
    name="BLS CPS National Unemployment (demographics)",
    url="https://api.bls.gov/publicAPI/v2/timeseries/data/",
    license="Public domain (U.S. government work)",
    notes=(
        "Seasonally adjusted monthly unemployment rates by age/sex/race + U3-U6 "
        "alternative measures. 18 of 20 requested series returned data. "
        "Missing: 55-64y (LNS14000095) and 65+y (LNS14000097) — series IDs may "
        "have changed in recent BLS catalog updates."
    ),
    methodology=(
        "BLS CPS household survey, ~60,000 households/month. U-3 = official "
        "unemployment rate (jobless + available + actively searched last 4 weeks "
        "/ civilian labor force). U-6 = U-3 + marginally attached + involuntary "
        "part-timers. All series seasonally adjusted via X-13ARIMA-SEATS."
    ),
    series_breaks=(
        "U-4/U-5/U-6 begin 1994. Black series 1972, Hispanic 1973, Asian 2003, "
        "White 1954. Biennial CPS redesigns (1994 most significant) introduced "
        "new questions affecting duration/reason categories — rates themselves "
        "are broadly comparable across the full history."
    ),
)
print("Source registered in _sources.")


In [ ]:
# Keep series_catalog current: upsert the 20+ adult total label (idempotent).
con.execute("""
DELETE FROM series_catalog WHERE series_id = 'LNS14000024';
INSERT INTO series_catalog (series_id, label, category)
VALUES ('LNS14000024', 'Total, 20+ (adults)', 'demographic');
""")
print("series_catalog rows:", con.execute("SELECT COUNT(*) FROM series_catalog").fetchone()[0])
print(con.execute("SELECT * FROM series_catalog WHERE series_id='LNS14000024'").fetchall())

## State-Level Series (BLS LAUS, LASST prefix)

Source: **BLS Local Area Unemployment Statistics (LAUS)** program.

51 state-level series (50 states + DC), monthly, 1976–2026.
All seasonally adjusted unemployment rates (U-3 definition only at state level).

**Key difference from national CPS:** LAUS state estimates are **model-based**
(not direct survey reads). The state CPS sample is too small to publish alone,
so LAUS blends CPS with UI claims data and CES payroll estimates via a
signal-plus-noise time-series model. Re-estimation of the model means historical
values can be revised — treat each pull as a single vintage.


In [ ]:
# State data was fetched in a prior session and is already in DuckDB.
df_states = con.execute("SELECT * FROM unemployment_states").df()
print(f"unemployment_states: {len(df_states):,} rows")
print(f"States: {df_states['label'].nunique()}")
years = df_states["year"].agg(["min", "max"])
print(f"Year range: {int(df_states['year'].min())} – {int(df_states['year'].max())}")
print()
print(df_states.head(10).to_string(index=False))


## Unemployment by education × race — annual table cpsaat07

A **companion** data source for the race-gap story (Chart 3 in `04-viz`): the
Black/White unemployment gap persists **at every education level**, so it is not
explained by an education/skills gap. To chart that, we need the **race ×
education cross-tab** of the unemployment rate.

**Why not the BLS API (route checked, 2026-10-05):** the API carries
*education-only* seasonally-adjusted series (e.g. `LNS14027660` HS-no-college)
and *race-only* series (`LNS14000003` White, `LNS14000006` Black), but **not
their intersection**. BLS publishes the race × education cross-tab only via the
annual household-data table **cpsaat07** — "Employment status of the civilian
noninstitutional population **25 years and over** by educational attainment, sex,
race, and Hispanic or Latino ethnicity."

**Source:** `https://www.bls.gov/cps/data/aa2024/cpsaat07.pdf` (annual averages;
prior years at `…/aa<YEAR>/cpsaat07.pdf`). PDF, public domain.

**⚠ Grain is DIFFERENT from Charts 1–6.** This table is an **annual average**,
population **25 years and over**, **not seasonally adjusted** — versus the
**monthly, seasonally-adjusted, 16+** `LN` series behind the other charts. The two
must never be plotted on the same axis or implied to be the same series. The
companion chart's subtitle states "annual average · people 25 and older".

**⚠ Access gotcha:** every bls.gov `/cps/` endpoint returns **403 (Akamai bot
filter)** to a plain request. It is bypassed by sending a complete browser header
set (`Accept`, `Referer`, `Sec-Fetch-*`); `fetch_bls_cps_table()` does so. The raw
PDF lands in `data/raw/cpsaat07_<year>.pdf` **verbatim** (never modified).

The parser (`parse_cpsaat07`) reads the PDF by word x-position (no ruled borders),
locating the **White** and **Black** blocks' "Unemployment rate" rows and taking
the four education-tier columns — using the published **TOTAL** column for the two
higher tiers (ignoring their sub-splits). It asserts 8 numeric columns per row and
plausible 0–30 rates so a future BLS layout change fails loudly.

In [ ]:
from ingest import raw_path
# Fetch + parse cpsaat07 (race × education, 25+ annual averages).
# Raw PDF is cached verbatim to data/raw/; re-running is safe (overwrites the
# same raw file, then re-parses). We pull 2024; 2023 is available via the same
# aa<YEAR> path if a robustness check is wanted later.
from ingest import fetch_bls_cps_table, parse_cpsaat07

CPSAAT07_YEAR = 2024
_url = f"https://www.bls.gov/cps/data/aa{CPSAAT07_YEAR}/cpsaat07.pdf"
_pdf = raw_path(cfg, f"cpsaat07_{CPSAAT07_YEAR}.pdf")

fetch_bls_cps_table(_url, _pdf)
print(f"Raw PDF cached: {_pdf}  ({_pdf.stat().st_size:,} bytes)")

df_edu_race = parse_cpsaat07(_pdf, CPSAAT07_YEAR)
print(f"\nParsed {len(df_edu_race)} rows (2 races × 4 education tiers):\n")
print(df_edu_race.to_string(index=False))

## Quick inspection — plausibility checks


In [ ]:
# National: check headline U-3 total rate at well-known recession peaks
total = con.execute("""
    SELECT year, period, value
    FROM unemployment_national
    WHERE series_id = 'LNS14000000'
      AND (
          (year = 1982 AND period = 'M11') OR  -- early-80s recession peak ~10.8
          (year = 2009 AND period = 'M10') OR  -- Great Recession peak ~10.0
          (year = 2020 AND period = 'M04') OR  -- COVID peak ~14.7
          (year = 2026 AND period = 'M07')     -- recent
      )
    ORDER BY year, period
""").df()
print("Headline U-3 at recession peaks + recent:")
print(total.to_string(index=False))

# State: check current rate for a few known states
recent_states = con.execute("""
    SELECT label, year, period, value
    FROM unemployment_states
    WHERE year = (SELECT MAX(year) FROM unemployment_states)
    ORDER BY value DESC
    LIMIT 10
""").df()
print()
print("Top 10 states by unemployment rate (most recent year):")
print(recent_states.to_string(index=False))


## Summary

| Table | Rows | Coverage | Notes |
|---|---|---|---|
| `unemployment_national` | 14,731 | 1948–2026, 19 series | National by age/sex/race + U3–U6 (incl. 20+ adult total) |
| `unemployment_states` | 30,950 | 1976–2026, 51 series | All states + DC, U-3 only |
| `series_catalog` | 72 | — | Series ID → label lookup |
| `state_ref` | 51 | — | State FIPS + series ID crosswalk |

Raw data saved to `data/raw/bls_api_data.parquet` (national) and
`data/raw/bls_api_states.parquet` (states).

**Next:** `02-clean.ipynb` — standardize dates, run quality checks, save
analysis-ready interim Parquet files.


## Cleanup

In [ ]:
con.close()
print('Connection closed.')